# 30-Day Hospital Readmission: raw CSVs to final submission

Intro to Machine Learning, Fall 2026 (IBA Karachi). Predict `readmit_30d` (ROC-AUC).

**How to run on Kaggle:** open the competition, click *Code* > *New Notebook*, upload/paste this notebook, make sure the competition data is attached (right panel *Input*), then *Run All*. Leave *Internet* off (everything needed is preinstalled). The notebook finds the data by itself and writes `/kaggle/working/submission.csv`.

**What it does**
1. Inspects the raw tables and shows the data-quality problems.
2. Cleans the data and builds one row per stay: feature set v1 (300 columns), v2 (+18) and v3 (+145).
3. Compares models with the same 5-fold cross-validation (LightGBM, XGBoost, CatBoost, logistic regression, Naive Bayes, decision tree).
4. Tunes LightGBM regularisation and target encoding, builds the final rank blend and writes the submission.

Set `QUICK = True` below for a fast smoke test (2 folds, few rounds); the real run uses `QUICK = False`.

In [ ]:
import os, glob, json, re, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

QUICK = os.environ.get('QUICK', '0') == '1'      # True = fast smoke test, False = full run
cands = glob.glob('/kaggle/input/**/train.csv', recursive=True)
DATA = os.path.dirname(cands[0]) if cands else os.environ.get('DATA_DIR', 'data')
D = DATA + '/'
OUT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
print('data folder:', DATA, '| output folder:', OUT, '| QUICK =', QUICK)
print(sorted(os.listdir(DATA)))

In [ ]:
import gc, types
KEEP = {'QUICK', 'DATA', 'D', 'OUT', 'ie_raw', 'tr', 'te', 'y', 'X1', 'X2', 'X3', 'cands'}


def free_memory():
    """drop the big raw tables / temporaries of a stage; keeps only the feature tables and the core variables"""
    for k in list(globals()):
        if k.startswith('_') or k in KEEP or k in ('In', 'Out', 'get_ipython', 'exit', 'quit', 'free_memory', 'gc', 'types', 'KEEP'):
            continue
        if isinstance(globals()[k], types.ModuleType):
            continue
        del globals()[k]
    gc.collect()

## 1. Inspect the raw data

In [ ]:
ie_raw = pd.read_csv(D + 'index_encounters.csv'); tr = pd.read_csv(D + 'train.csv'); te = pd.read_csv(D + 'test.csv')
y = tr.readmit_30d.values
print('stays:', len(ie_raw), '| train:', len(tr), '| test:', len(te), '| positive rate in train: %.2f%%' % (100 * y.mean()))
rows = {}
for t in ['diagnoses', 'procedures', 'lab_results', 'vitals', 'medications', 'unit_transfers', 'discharge_planning']:
    d_ = pd.read_csv(D + t + '.csv')
    rows[t] = dict(rows=len(d_), stays=d_.encounter_id.nunique(), coverage='%.1f%%' % (100 * d_.encounter_id.nunique() / len(ie_raw)), exact_duplicates=int(d_.duplicated().sum()))
pu_ = pd.read_csv(D + 'prior_utilization.csv'); rows['prior_utilization'] = dict(rows=len(pu_), stays=pu_.patient_id.nunique(), coverage='(patients)', exact_duplicates=int(pu_.duplicated().sum()))
pd.DataFrame(rows).T

In [ ]:
# data-quality problems that the cleaning below fixes
lab_ = pd.read_csv(D + 'lab_results.csv'); vit_ = pd.read_csv(D + 'vitals.csv'); dx_ = pd.read_csv(D + 'diagnoses.csv'); med_ = pd.read_csv(D + 'medications.csv')
print('lab units per test (same test, different units):')
print(lab_.groupby(['lab_name', 'unit']).size().unstack().loc[['Albumin', 'Creatinine', 'Glucose', 'Hemoglobin']].dropna(axis=1, how='all').fillna(0).astype(int))
print('\nTEMP readings above 45 (Fahrenheit):', int(((vit_.vital_type == 'TEMP') & (vit_.value > 45)).sum()), 'of', int((vit_.vital_type == 'TEMP').sum()))
print('ICD codes without a decimal point: %.1f%%' % (100 * (~dx_.icd10_code.str.contains(r'\.')).mean()))
print('drug-name spellings:', med_.drug_name.nunique(), '-> after lower-casing/trimming:', med_.drug_name.str.lower().str.strip().nunique())
print('placeholders:', (pd.read_csv(D + 'patients.csv').race_ethnicity == '?').sum(), "race '?',", int(ie_raw.admission_type.isin(['Not Available', 'Not Mapped']).sum()), 'admission_type placeholders')
del lab_, vit_, dx_, med_, pu_, d_

## 2. Cleaning and feature engineering

Three stages build one row per `encounter_id`. Cleaning rules: one unit per lab test, Fahrenheit temperatures converted, implausible values set to missing, duplicates dropped, ICD codes given their missing decimal point, placeholders turned into missing, spellings normalised.

### 2a. Feature set v1: base tables, diagnoses, procedures, labs, vitals, medications, transfers, discharge planning, prior visits

**setup**

In [ ]:
import pandas as pd, numpy as np
def rd(n,**k): return pd.read_csv(D+n+'.csv',**k)

**base table: index encounters + patients + hospitals**

In [ ]:
ie=rd('index_encounters',parse_dates=['admit_dt','discharge_dt'])
pt=rd('patients'); hp=rd('hospitals')
pt['insurance_type']=pt.insurance_type.str.title()          # MEDICARE -> Medicare
pt['race_ethnicity']=pt.race_ethnicity.replace('?',np.nan)
pt['sex']=pt.sex.replace('U',np.nan)
pt.loc[pt.birth_year<1910,'birth_year']=np.nan
for c in ['admission_type','admission_source','discharge_disposition']:
    ie[c]=ie[c].replace({'Not Available':np.nan,'Not Mapped':np.nan})
df=ie.merge(pt,on='patient_id',how='left').merge(hp,on='hospital_id',how='left')
df['los_days']=(df.discharge_dt-df.admit_dt).dt.total_seconds()/86400
df.loc[(df.los_days<=0)|(df.los_days>120),'los_days']=np.nan
df['age']=df.admit_dt.dt.year-df.birth_year
df['admit_hour']=df.admit_dt.dt.hour; df['admit_dow']=df.admit_dt.dt.dayofweek
df['admit_month']=df.admit_dt.dt.month; df['admit_year']=df.admit_dt.dt.year
df['weight_missing']=df.admit_weight_kg.isna().astype(int)
df['spec_missing']=df.attending_specialty.isna().astype(int)
df['attending_n']=df.groupby('attending_id').encounter_id.transform('count')
df['hosp_n']=df.groupby('hospital_id').encounter_id.transform('count')
df['zip3']=df.zip3.astype('float')

**diagnoses**

In [ ]:
dx=rd('diagnoses').drop_duplicates(['encounter_id','seq'])
def fix(c):
    c=c.strip().upper()
    return c if '.' in c else (c[:3]+'.'+c[3:] if len(c)>3 else c)
dx['icd10_code']=dx.icd10_code.map(fix)
lk=rd('icd10_lookup'); dx=dx.merge(lk[['icd10_code','chapter','chronic_flag']],on='icd10_code',how='left')
dx['poa_n']=(dx.present_on_admission=='N').astype(int)
g=dx.groupby('encounter_id')
f_dx=pd.DataFrame({'dx_n':g.size(),'dx_chronic_n':g.chronic_flag.sum(),'dx_poa_n':g.poa_n.sum(),'dx_poa_missing':g.present_on_admission.apply(lambda s:s.isna().sum())})
prim=dx[dx.is_primary==1].drop_duplicates('encounter_id').set_index('encounter_id')
f_dx['dx_primary']=prim.icd10_code; f_dx['dx_primary_chapter']=prim.chapter
f_dx['dx_primary3']=prim.icd10_code.str[:3]
ch=pd.crosstab(dx.encounter_id,dx.chapter).add_prefix('dxch_'); f_dx=f_dx.join(ch)
d3=dx.assign(c3=dx.icd10_code.str[:3])
top=d3.c3.value_counts().head(60).index
f_dx=f_dx.join(pd.crosstab(d3[d3.c3.isin(top)].encounter_id,d3[d3.c3.isin(top)].c3).clip(upper=1).add_prefix('dx3_'))

**procedures**

In [ ]:
pr=rd('procedures').drop_duplicates(); pr['code']=pr.code.astype(str)
f_pr=pr.groupby('encounter_id').agg(proc_n=('code','size'),proc_first=('code','first'))

**labs: unit harmonisation + plausibility**

In [ ]:
lab=rd('lab_results').drop_duplicates()
lab['collected_dt']=pd.to_datetime(lab.collected_dt,errors='coerce')
conv={('Albumin','g/L'):.1,('Creatinine','umol/L'):1/88.4,('Glucose','mmol/L'):18.016,('Hemoglobin','g/L'):.1}
fac=[conv.get((n,u),1.) for n,u in zip(lab.lab_name,lab.unit)]
lab['v']=lab.value*np.array(fac)
rng={'Albumin':(1,6),'Bnp':(5,40000),'Bun':(2,300),'Creatinine':(.2,20),'Glucose':(30,1000),'Hba1C':(3,18),'Hemoglobin':(3,22),'Inr':(.7,15),'Lactate':(.2,30),'Potassium':(1.5,9),'Sodium':(100,180),'Troponin_I':(0,50),'Wbc':(.5,100)}
lo=lab.lab_name.map(lambda x:rng[x][0]); hi=lab.lab_name.map(lambda x:rng[x][1])
lab.loc[(lab.v<lo)|(lab.v>hi),'v']=np.nan            # unit-slip / typo outliers -> missing
lab=lab.sort_values(['encounter_id','collected_dt'])
g=lab.groupby(['encounter_id','lab_name']).v
f_lab=pd.concat({'last':g.last(),'mean':g.mean(),'min':g.min(),'max':g.max(),'first':g.first(),'cnt':g.count()},axis=1).unstack('lab_name')
f_lab.columns=[f'lab_{b}_{a}' for a,b in f_lab.columns]
for l in rng: f_lab[f'lab_{l}_delta']=f_lab[f'lab_{l}_last']-f_lab[f'lab_{l}_first']
f_lab=f_lab.drop(columns=[c for c in f_lab if c.endswith('_first')])
ab=pd.crosstab(lab.encounter_id,lab.abnormal_flag)
f_lab['lab_abn_H']=ab.get('H'); f_lab['lab_abn_L']=ab.get('L'); f_lab['lab_total']=lab.groupby('encounter_id').size()

**vitals**

In [ ]:
v=rd('vitals').drop_duplicates()
v['t']=pd.to_datetime(v.charttime,format='%m/%d/%Y %H:%M',errors='coerce')
v.loc[(v.vital_type=='TEMP')&(v.value>45),'value']=(v.value-32)*5/9      # Fahrenheit -> Celsius
r={'HR':(20,250),'SBP':(50,260),'DBP':(20,160),'RR':(4,60),'SPO2':(50,100),'TEMP':(30,43)}
lo=v.vital_type.map(lambda x:r[x][0]); hi=v.vital_type.map(lambda x:r[x][1])
v.loc[(v.value<lo)|(v.value>hi),'value']=np.nan
v=v.sort_values(['encounter_id','t'])
g=v.groupby(['encounter_id','vital_type']).value
f_v=pd.concat({'last':g.last(),'first':g.first(),'mean':g.mean(),'min':g.min(),'max':g.max(),'std':g.std(),'cnt':g.count()},axis=1).unstack('vital_type')
f_v.columns=[f'vit_{b}_{a}' for a,b in f_v.columns]
for k in r: f_v[f'vit_{k}_delta']=f_v[f'vit_{k}_last']-f_v[f'vit_{k}_first']
f_v=f_v.drop(columns=[c for c in f_v if c.endswith('_first')])

**medications**

In [ ]:
m=rd('medications').drop_duplicates()
m['s']=pd.to_datetime(m.start_dt.astype(str),format='%Y%m%d',errors='coerce'); m['e']=pd.to_datetime(m.stop_dt.astype('Int64').astype(str),format='%Y%m%d',errors='coerce')
g=m.groupby('encounter_id')
f_m=pd.DataFrame({'med_n':g.size(),'med_distinct':g.drug_name.nunique(),'med_cont_n':g.continued_at_discharge.sum(),'med_cont_missing':g.continued_at_discharge.apply(lambda s:s.isna().sum()),
  'med_iv_n':g.route.apply(lambda s:(s=='IV').sum()),'med_nostop_n':g.stop_dt.apply(lambda s:s.isna().sum())})
f_m=f_m.join(pd.crosstab(m.encounter_id,m.drug_class).clip(upper=1).add_prefix('med_'))

**unit transfers**

In [ ]:
u=rd('unit_transfers',parse_dates=['in_dt','out_dt']).drop_duplicates()
u['h']=(u.out_dt-u.in_dt).dt.total_seconds()/3600; u.loc[(u.h<0),'h']=np.nan
f_u=u.groupby('encounter_id').agg(tr_n=('unit','size'),tr_last_unit=('unit','last'),tr_first_unit=('unit','first'))
f_u=f_u.join(u.pivot_table(index='encounter_id',columns='unit',values='h',aggfunc='sum').add_prefix('tr_hours_'))
f_u['tr_icu']=(u.assign(i=u.unit=='ICU').groupby('encounter_id').i.max()).astype(int)

**discharge planning**

In [ ]:
dp=rd('discharge_planning').drop_duplicates('encounter_id').set_index('encounter_id')
dp['home_health_referral']=dp.home_health_referral.map({'Y':1,'N':0}); dp['education_done']=dp.education_done.map({'Yes':1,'No':0})
dp['dp_present']=1; dp['followup_missing']=dp.followup_scheduled_days.isna().astype(int)
dp.loc[dp.followup_scheduled_days>60,'followup_scheduled_days']=np.nan

**prior utilization (only visits before admit)**

In [ ]:
pu=rd('prior_utilization',parse_dates=['visit_dt']).drop_duplicates()
pu=pu.merge(df[['patient_id','admit_dt']],on='patient_id'); pu['d']=(pu.admit_dt-pu.visit_dt).dt.days
pu=pu[pu.d>0]
rows={}
for t in ['ED','IP','OP']:
    s=pu[pu.visit_type==t]
    for w in [30,90,180,365]: rows[f'pu_{t}_{w}d']=s[s.d<=w].groupby('patient_id').size()
    rows[f'pu_{t}_all']=s.groupby('patient_id').size(); rows[f'pu_{t}_lastgap']=s.groupby('patient_id').d.min()
rows['pu_all']=pu.groupby('patient_id').size(); rows['pu_nhosp']=pu.groupby('patient_id').hospital_id.nunique()
f_pu=pd.DataFrame(rows)

**assemble**

In [ ]:
X=df.set_index('encounter_id')
for f in [f_dx,f_pr,f_lab,f_v,f_m,f_u,dp]: X=X.join(f)
X=X.join(f_pu,on='patient_id')
cnt=[c for c in X if c.startswith(('pu_','dx_n','proc_n','med_n')) and 'lastgap' not in c]
for c in [c for c in X if c.startswith(('dxch_','dx3_','med_')) or c in('proc_n','pu_all')]: X[c]=X[c].fillna(0)
for c in [c for c in X if c.startswith('pu_') and 'lastgap' not in c]: X[c]=X[c].fillna(0)
X['dp_present']=X.dp_present.fillna(0)
X['pu_acute_365']=X.pu_ED_365d+X.pu_IP_365d
X['pu_acute_90']=X.pu_ED_90d+X.pu_IP_90d
X=X.drop(columns=['patient_id','admit_dt','discharge_dt','birth_year','dx_primary'],errors='ignore')
cats=[c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c]) and not pd.api.types.is_datetime64_any_dtype(X[c])]
for c in cats: X[c]=X[c].astype('category')
X1 = X.copy()
print(X.shape, len(cats),'categoricals'); print(cats)

In [ ]:
free_memory(); print(X1.shape)

### 2b. Feature set v2: Charlson and LACE-style scores, ratios, extra diagnosis flags, outlier counts

**setup**

In [ ]:
"""v2 features: adds comorbidity/LACE scores, wider diagnosis flags, ratios and outlier flags to the v1 table."""
import numpy as np
import pandas as pd

X = X1.copy()

**diagnoses: Charlson-style score + extra 3-char flags**

In [ ]:
dx = pd.read_csv(D + 'diagnoses.csv').drop_duplicates(['encounter_id', 'seq'])
dx['icd'] = dx.icd10_code.str.strip().str.upper()
dx['c3'] = dx.icd.str[:3]

charlson = {  # prefix -> weight (simplified Charlson)
    'I21': 1, 'I22': 1, 'I50': 1, 'I70': 1, 'I73': 1, 'G45': 1, 'F00': 1, 'F01': 1, 'F02': 1, 'F03': 1,
    'G30': 1, 'J40': 1, 'J41': 1, 'J42': 1, 'J43': 1, 'J44': 1, 'J45': 1, 'J47': 1, 'M05': 1, 'M06': 1,
    'M32': 1, 'K25': 1, 'K26': 1, 'K27': 1, 'K28': 1, 'K70': 1, 'K74': 1, 'E10': 1, 'E11': 1,
    'G81': 2, 'G82': 2, 'N18': 2, 'N19': 2, 'B20': 6}
w = dx.c3.map(charlson).fillna(0)
w[dx.c3.str[0].eq('C') & dx.c3.between('C00', 'C76')] = 2
w[dx.c3.between('C77', 'C80')] = 6
dx['w'] = w
ch = dx.groupby('encounter_id').w.sum().rename('charlson')

# flags for the 150 most common 3-char codes (v1 had 60)
top = dx.c3.value_counts().head(150).index
fl = pd.crosstab(dx[dx.c3.isin(top)].encounter_id, dx[dx.c3.isin(top)].c3).clip(upper=1).add_prefix('dx3b_')
# only add the ones v1 did not have
fl = fl[[c for c in fl if 'dx3_' + c[5:] not in X.columns]]

# raw-code count of distinct 3-char codes and share of chronic
nd = dx.groupby('encounter_id').c3.nunique().rename('dx_distinct3')

X = X.join(ch).join(nd).join(fl)
X['charlson'] = X.charlson.fillna(0)
for c in fl.columns:
    X[c] = X[c].fillna(0)

**LACE-style score**

In [ ]:
los = X.los_days
L = pd.cut(los, [-1, 1, 2, 3, 6, 13, 1000], labels=[0, 1, 2, 3, 4, 5], right=False).astype(float)
L = np.select([los < 1, los < 2, los < 3, los < 4, los < 7, los < 14], [0, 1, 2, 3, 4, 5], default=7)
A = (X.admission_type.astype(str) == 'Emergency').astype(int) * 3
C = np.select([X.charlson == 0, X.charlson == 1, X.charlson == 2, X.charlson == 3], [0, 1, 2, 3], default=5)
E = np.minimum(X.pu_ED_180d.fillna(0), 4)
X['lace'] = L + A + C + E

**ratios / interactions**

In [ ]:
X['bun_creat'] = X.lab_Bun_last / X.lab_Creatinine_last
X['med_cont_frac'] = X.med_cont_n / X.med_n.replace(0, np.nan)
X['med_per_day'] = X.med_n / X.los_days
X['dx_per_day'] = X.dx_n / X.los_days
X['pu_acute_frac'] = X.pu_acute_365 / X.pu_all.replace(0, np.nan)
X['prior_ip_recent'] = (X.pu_IP_lastgap <= 30).astype(int)
X['age_x_charlson'] = X.age * X.charlson
X['los_x_charlson'] = X.los_days * X.charlson
X['icu_hours'] = X.get('tr_hours_ICU')
X['shock_index'] = X.vit_HR_last / X.vit_SBP_last

**outlier flags: how many raw lab values fell outside plausible range**

In [ ]:
lab = pd.read_csv(D + 'lab_results.csv').drop_duplicates()
conv = {('Albumin', 'g/L'): .1, ('Creatinine', 'umol/L'): 1 / 88.4, ('Glucose', 'mmol/L'): 18.016, ('Hemoglobin', 'g/L'): .1}
lab['v'] = lab.value * np.array([conv.get((n, u), 1.) for n, u in zip(lab.lab_name, lab.unit)])
rng = {'Albumin': (1, 6), 'Bnp': (5, 40000), 'Bun': (2, 300), 'Creatinine': (.2, 20), 'Glucose': (30, 1000), 'Hba1C': (3, 18),
       'Hemoglobin': (3, 22), 'Inr': (.7, 15), 'Lactate': (.2, 30), 'Potassium': (1.5, 9), 'Sodium': (100, 180),
       'Troponin_I': (0, 50), 'Wbc': (.5, 100)}
lo = lab.lab_name.map(lambda x: rng[x][0]); hi = lab.lab_name.map(lambda x: rng[x][1])
lab['out'] = ((lab.v < lo) | (lab.v > hi)).astype(int)
X = X.join(lab.groupby('encounter_id').out.sum().rename('lab_outlier_n'))
X['lab_outlier_n'] = X.lab_outlier_n.fillna(0)
X = X.join(lab.groupby('encounter_id').lab_name.nunique().rename('lab_distinct_tests'))

X = X.copy()
X2 = X.copy()
print(X.shape)
print(X[['charlson', 'lace']].describe().T)

In [ ]:
free_memory(); print(X2.shape)

### 2c. Feature set v3: state at discharge (lab/vital timing and trends), drug names, procedure and transfer timing, same-hospital prior visits

**setup**

In [ ]:
"""v3 features on top of X2.pkl: the patient's state at discharge and timing information.
  l3_*  labs:        hours between last lab and discharge, last-24h counts, abnormal flag of the last value, per-test slope
  v3_*  vitals:      last-24h mean/min/max per vital, clinical-threshold abnormal counts near discharge
  m3_*  medications: drug-name flags (names normalised: 122 spellings -> 52), started/stopped timing, high-risk drugs
  p3_*  procedures:  timing relative to admission/discharge (CPT and CPT-4 treated as one system)
  t3_*  transfers:   timing of the last unit move and of the last ICU stay relative to discharge
  u3_*  prior visits: split into same hospital as the index stay vs other hospitals
"""
import json
import re
import time

import numpy as np
import pandas as pd

t0 = time.time()


def log(msg):
    print(f'[{round(time.time() - t0)}s] {msg}', flush=True)


def clean(s):
    return re.sub(r'[^0-9a-zA-Z]+', '_', str(s).strip().lower()).strip('_')


X = X2.copy()
ie = pd.read_csv(D + 'index_encounters.csv', parse_dates=['admit_dt', 'discharge_dt'])[
    ['encounter_id', 'patient_id', 'hospital_id', 'admit_dt', 'discharge_dt']]
NEW = {}  # group prefix -> list of DataFrames


def add(prefix, df):
    NEW.setdefault(prefix, []).append(df)

**labs**

In [ ]:
lab = pd.read_csv(D + 'lab_results.csv').drop_duplicates()
lab['collected_dt'] = pd.to_datetime(lab.collected_dt, errors='coerce')
conv = {('Albumin', 'g/L'): .1, ('Creatinine', 'umol/L'): 1 / 88.4, ('Glucose', 'mmol/L'): 18.016, ('Hemoglobin', 'g/L'): .1}
lab['v'] = lab.value * np.array([conv.get((n, u), 1.) for n, u in zip(lab.lab_name, lab.unit)])
rng = {'Albumin': (1, 6), 'Bnp': (5, 40000), 'Bun': (2, 300), 'Creatinine': (.2, 20), 'Glucose': (30, 1000), 'Hba1C': (3, 18),
       'Hemoglobin': (3, 22), 'Inr': (.7, 15), 'Lactate': (.2, 30), 'Potassium': (1.5, 9), 'Sodium': (100, 180),
       'Troponin_I': (0, 50), 'Wbc': (.5, 100)}
lo = lab.lab_name.map(lambda x: rng[x][0]); hi = lab.lab_name.map(lambda x: rng[x][1])
lab.loc[(lab.v < lo) | (lab.v > hi), 'v'] = np.nan
lab = lab.merge(ie[['encounter_id', 'admit_dt', 'discharge_dt']], on='encounter_id')
lab['hb'] = (lab.discharge_dt - lab.collected_dt).dt.total_seconds() / 3600       # hours before discharge
lab['ta'] = (lab.collected_dt - lab.admit_dt).dt.total_seconds() / 3600          # hours since admission
lab['l24'] = (lab.hb <= 24).astype(int)
lab['abn'] = lab.abnormal_flag.isin(['H', 'L']).astype(int)
g = lab.groupby('encounter_id')
f = pd.DataFrame({'l3_hrs_last_lab': g.hb.min(), 'l3_hrs_mean_lab': g.hb.mean(), 'l3_n_last24': g.l24.sum(),
                  'l3_frac_last24': g.l24.mean(), 'l3_abn_last24': (lab.abn * lab.l24).groupby(lab.encounter_id).sum()})
add('l3', f)
lab = lab.sort_values(['encounter_id', 'lab_name', 'collected_dt'])
gl = lab.groupby(['encounter_id', 'lab_name'])
f = gl.hb.min().unstack(); f.columns = [f'l3_{clean(c)}_hrs_since' for c in f.columns]; add('l3', f)
flag = lab.assign(fl=lab.abnormal_flag.map({'H': 1, 'L': -1, 'N': 0})).groupby(['encounter_id', 'lab_name']).fl.last().unstack()
flag.columns = [f'l3_{clean(c)}_lastflag' for c in flag.columns]; add('l3', flag)
lv = lab.dropna(subset=['v']).copy(); lv['tt'] = lv.ta ** 2; lv['tv'] = lv.ta * lv.v
s = lv.groupby(['encounter_id', 'lab_name']).agg(n=('v', 'size'), st=('ta', 'sum'), sv=('v', 'sum'), stt=('tt', 'sum'), stv=('tv', 'sum'))
den = s.n * s.stt - s.st ** 2
slope = ((s.n * s.stv - s.st * s.sv) / den).where((s.n >= 3) & (den > 1e-6))
slope = slope.unstack(); slope.columns = [f'l3_{clean(c)}_slope' for c in slope.columns]; add('l3', slope)
del lab, lv, s
log('labs done')

**vitals**

In [ ]:
v = pd.read_csv(D + 'vitals.csv').drop_duplicates()
v['t'] = pd.to_datetime(v.charttime, format='%m/%d/%Y %H:%M', errors='coerce')
v.loc[(v.vital_type == 'TEMP') & (v.value > 45), 'value'] = (v.value - 32) * 5 / 9
r = {'HR': (20, 250), 'SBP': (50, 260), 'DBP': (20, 160), 'RR': (4, 60), 'SPO2': (50, 100), 'TEMP': (30, 43)}
lo = v.vital_type.map(lambda x: r[x][0]); hi = v.vital_type.map(lambda x: r[x][1])
v.loc[(v.value < lo) | (v.value > hi), 'value'] = np.nan
v = v.merge(ie[['encounter_id', 'discharge_dt']], on='encounter_id')
v['hb'] = (v.discharge_dt - v.t).dt.total_seconds() / 3600
x, vt = v.value, v.vital_type
v['abn'] = (((vt == 'HR') & ((x > 100) | (x < 50))) | ((vt == 'SBP') & ((x < 90) | (x > 180))) | ((vt == 'SPO2') & (x < 92))
            | ((vt == 'TEMP') & (x > 38)) | ((vt == 'RR') & (x > 24))).astype(int)
v['l24'] = (v.hb <= 24).astype(int); v['l12'] = (v.hb <= 12).astype(int)
g = v.groupby('encounter_id')
f = pd.DataFrame({'v3_hrs_last_vital': g.hb.min(), 'v3_abn_total': g.abn.sum(), 'v3_abn_frac': g.abn.mean(), 'v3_n_last24': g.l24.sum(),
                  'v3_abn_last24': (v.abn * v.l24).groupby(v.encounter_id).sum(), 'v3_abn_last12': (v.abn * v.l12).groupby(v.encounter_id).sum()})
f['v3_abn_last24_frac'] = f.v3_abn_last24 / f.v3_n_last24.replace(0, np.nan); add('v3', f)
w = v[v.l24 == 1].groupby(['encounter_id', 'vital_type']).value.agg(['mean', 'min', 'max']).unstack()
w.columns = [f'v3_{b}_l24_{a}' for a, b in w.columns]; add('v3', w)
del v
log('vitals done')

**medications**

In [ ]:
m = pd.read_csv(D + 'medications.csv')
m['dn'] = m.drug_name.map(clean)
m = m.drop_duplicates(['encounter_id', 'dn', 'start_dt', 'route'])
m['s'] = pd.to_datetime(m.start_dt.astype(str), format='%Y%m%d', errors='coerce')
m['e'] = pd.to_datetime(m.stop_dt.astype('Int64').astype(str), format='%Y%m%d', errors='coerce')
m = m.merge(ie[['encounter_id', 'admit_dt', 'discharge_dt']], on='encounter_id')
dcd, adm = m.discharge_dt.dt.normalize(), m.admit_dt.dt.normalize()
m['st_last1'] = (m.s >= dcd - pd.Timedelta(days=1)).astype(int)
m['st_adm'] = (m.s <= adm).astype(int)
m['stopped'] = (m.e < dcd).astype(int)
m['dur'] = (m.e - m.s).dt.days
hr = m.dn.str.contains('enoxaparin|heparin|warfarin|insulin|morphine|oxycodone|hydromorphone|fentanyl|digoxin|tramadol', regex=True)
m['hr'] = hr.astype(int)
g = m.groupby('encounter_id')
f = pd.DataFrame({'m3_started_last1d': g.st_last1.sum(), 'm3_started_admit_day': g.st_adm.sum(), 'm3_stopped_before_dc': g.stopped.sum(),
                  'm3_mean_dur_days': g.dur.mean(), 'm3_n_highrisk': g.hr.sum(), 'm3_n_names': g.dn.nunique()}); add('m3', f)
top = m.dn.value_counts().head(50).index
fl = pd.crosstab(m[m.dn.isin(top)].encounter_id, m[m.dn.isin(top)].dn).clip(upper=1); fl.columns = [f'm3_drug_{c}' for c in fl.columns]; add('m3', fl)
cont = m[m.continued_at_discharge == 1].groupby('encounter_id').dn.nunique().rename('m3_n_cont_names'); add('m3', cont.to_frame())
del m
log('meds done')

**procedures**

In [ ]:
p = pd.read_csv(D + 'procedures.csv').drop_duplicates()
p['d'] = pd.to_datetime(p.proc_dt, format='%d-%b-%Y', errors='coerce')
p = p.merge(ie[['encounter_id', 'admit_dt', 'discharge_dt']], on='encounter_id')
p['da'] = (p.d - p.admit_dt.dt.normalize()).dt.days; p['db'] = (p.discharge_dt.dt.normalize() - p.d).dt.days
p['code'] = p.code.astype(str).str.strip()
g = p.groupby('encounter_id')
f = pd.DataFrame({'p3_first_day': g.da.min(), 'p3_last_day': g.da.max(), 'p3_last_days_before_dc': g.db.min(),
                  'p3_any_last48h': (p.db <= 2).groupby(p.encounter_id).max().astype(int), 'p3_distinct_codes': g.code.nunique()}); add('p3', f)
del p
log('procedures done')

**unit transfers**

In [ ]:
u = pd.read_csv(D + 'unit_transfers.csv', parse_dates=['in_dt', 'out_dt']).drop_duplicates()
u = u.merge(ie[['encounter_id', 'discharge_dt']], on='encounter_id').sort_values(['encounter_id', 'in_dt'])
u['in_hb'] = (u.discharge_dt - u.in_dt).dt.total_seconds() / 3600; u['out_hb'] = (u.discharge_dt - u.out_dt).dt.total_seconds() / 3600
u['dur'] = (u.out_dt - u.in_dt).dt.total_seconds() / 3600
icu = u.unit.astype(str).str.upper().str.contains('ICU')
g = u.groupby('encounter_id')
f = pd.DataFrame({'t3_last_in_hrs_before_dc': g.in_hb.min(), 'm_tmp': g.dur.last(), 'tmp_n24': (u.in_hb <= 24).groupby(u.encounter_id).sum()})
f = f.rename(columns={'m_tmp': 't3_last_unit_hours', 'tmp_n24': 't3_n_moves_last24'})
f['t3_hrs_since_icu_out'] = u[icu].groupby('encounter_id').out_hb.min()
f['t3_icu_last24'] = ((u.out_hb <= 24) & icu).groupby(u.encounter_id).max().astype(int); add('t3', f)
del u
log('transfers done')

**prior visits: same hospital vs elsewhere**

In [ ]:
pu = pd.read_csv(D + 'prior_utilization.csv', parse_dates=['visit_dt']).drop_duplicates()
pu = pu.merge(ie[['patient_id', 'hospital_id', 'admit_dt']].rename(columns={'hospital_id': 'idx_h'}), on='patient_id')
pu['d'] = (pu.admit_dt - pu.visit_dt).dt.days; pu = pu[pu.d > 0]
pu['same'] = (pu.hospital_id == pu.idx_h).astype(int)
cols = {}
for tag, mask in [('same', pu.same == 1), ('diff', pu.same == 0)]:
    for w in [90, 365, 10 ** 6]:
        s_ = pu[mask & (pu.d <= w)].groupby('patient_id').size(); cols[f'u3_{tag}_{"all" if w > 1000 else w}d'] = s_
    cols[f'u3_{tag}_ip_all'] = pu[mask & (pu.visit_type == 'IP')].groupby('patient_id').size()
f = pd.DataFrame(cols).fillna(0)
f['u3_same_frac'] = f.u3_same_alld / (f.u3_same_alld + f.u3_diff_alld).replace(0, np.nan)
f = f.rename(columns={'u3_same_alld': 'u3_same_all', 'u3_diff_alld': 'u3_diff_all'})
log('prior done')

**assemble**

In [ ]:
X3 = X.copy(); groups = {}
for pre, dfs in NEW.items():
    d = pd.concat(dfs, axis=1)
    X3 = X3.join(d); groups[pre] = list(d.columns)
X3 = X3.join(ie.set_index('encounter_id')[['patient_id']].join(f, on='patient_id').drop(columns='patient_id'))
groups['u3'] = list(f.columns)
zero = [c for c in X3.columns if c.startswith(('u3_', 'm3_drug_', 'm3_n', 'm3_started', 'm3_stopped', 'v3_n_', 'v3_abn', 'l3_n_', 'l3_abn_', 't3_n_', 't3_icu_', 'p3_any'))]
for c in zero:
    if c in X3 and c != 'u3_same_frac':
        X3[c] = X3[c].fillna(0)
X3 = X3.copy()
log(f'X3 {X3.shape} (X2 had {X.shape[1]} cols); new per group: ' + str({k: len(v) for k, v in groups.items()}))
print(X3[[c for c in X3 if c[:3] in ('l3_', 'v3_', 'm3_', 'p3_', 't3_', 'u3_')]].isna().mean().round(2).describe())

In [ ]:
free_memory()

In [ ]:
print('final modelling table: X3 =', X3.shape, '| v2 table: X2 =', X2.shape, '| v1 table: X1 =', X1.shape)
print('numeric columns:', sum(str(X3[c].dtype) != 'category' for c in X3), '| categorical:', sum(str(X3[c].dtype) == 'category' for c in X3))

## 3. Validation setup and helpers

* 5-fold stratified CV on the training stays, same folds for every model.
* Target encoding (hospital, attending, primary diagnosis, first procedure code, ...) is fitted inside each training fold with an inner 5-fold scheme, so no row sees its own label.
* The imputers, scalers and encoders for the non-tree models are fitted on the training part of each fold only.

In [ ]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool
from scipy.stats import rankdata
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

folds = list(StratifiedKFold(5, shuffle=True, random_state=0).split(tr, y))
if QUICK:
    folds = folds[:2]
CAP = 200 if QUICK else 3000          # max boosting rounds (early stopping decides the real number)
SEEDS = 1 if QUICK else 3
results = {}                           # model name -> CV AUC


def cv_auc(oof):
    ok = ~np.isnan(oof)
    return roc_auc_score(y[ok], oof[ok])


# ---- target encoding (smoothed mean of the label per category) ----
TE_BASE = ['hospital_id', 'attending_id', 'dx_primary3', 'proc_first']
TE_EXT = TE_BASE + ['service_line', 'admission_source', 'discharge_disposition', 'dx_primary_chapter', 'attending_specialty', 'insurance_type']
M_SMOOTH = 50


def te_fit(col, yv):
    g = yv.mean()
    s = pd.DataFrame({'c': col.astype(object).values, 'y': yv}).groupby('c').y.agg(['sum', 'count'])
    return ((s['sum'] + M_SMOOTH * g) / (s['count'] + M_SMOOTH)), g


def te_apply(col, mp):
    m, g = mp
    return col.astype(object).map(m).astype(float).fillna(g).values


def add_te(Xtr_, ytr, others, cols, seed=0):
    """out-of-fold target encoding for the training part, full-train encoding for the others"""
    Xtr_ = Xtr_.copy(); outs = [o.copy() for o in others]
    for c in cols:
        oof = np.zeros(len(Xtr_))
        for a, b in StratifiedKFold(5, shuffle=True, random_state=seed).split(Xtr_, ytr):
            oof[b] = te_apply(Xtr_[c].iloc[b], te_fit(Xtr_[c].iloc[a], ytr[a]))
        mp = te_fit(Xtr_[c], ytr); Xtr_['te_' + c] = oof
        for o in outs:
            o['te_' + c] = te_apply(o[c], mp)
    return Xtr_, outs


def prep(A):
    """high-cardinality identifiers are used through their target encoding, not raw"""
    return A.drop(columns=['attending_id', 'proc_first'], errors='ignore')


_cache = {}


def to_f32(df):
    return df.astype({c: 'float32' for c in df.select_dtypes('number').columns})


def fold_data(Xfull, key, cols):
    """target-encode once per fold and reuse for every LightGBM setting (cached per feature set and column list)"""
    key = key.split('_')[0] + '|' + ','.join(cols)
    if key not in _cache:
        Xt = Xfull.loc[tr.encounter_id]; out = []
        for a, b in folds:
            A, (B,) = add_te(Xt.iloc[a], y[a], [Xt.iloc[b]], cols); A, B = prep(A), prep(B)
            out.append((to_f32(A), to_f32(B)) if QUICK else (A, B))   # float32 only in the smoke test (saves memory)
        _cache[key] = out
    return _cache[key]


BASE_P = dict(objective='binary', learning_rate=0.04, num_leaves=4, min_child_samples=400, feature_fraction=0.2, bagging_fraction=0.8,
              bagging_freq=1, lambda_l2=60, cat_smooth=50, min_data_per_group=100, verbose=-1, n_jobs=-1)


def cv_lgb(params, key, Xfull, cols=TE_BASE):
    oof = np.full(len(y), np.nan); its = []
    for (A, B), (a, b) in zip(fold_data(Xfull, key, cols), folds):
        m = lgb.train(params, lgb.Dataset(A, y[a]), CAP, valid_sets=[lgb.Dataset(B, y[b])], callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[b] = m.predict(B); its.append(m.best_iteration)
    return cv_auc(oof), oof, int(np.mean(its))

## 4. Compare models (same folds)

In [ ]:
# --- LightGBM with the v2 setting (strongly regularised: 4 leaves, 400 rows per leaf, L2 = 60) ---
auc, oof_lgb2, its2 = cv_lgb(BASE_P, 'X2_base', X2); results['LightGBM (v2 features)'] = auc; print('LightGBM v2 features: %.4f  (best rounds %d)' % (auc, its2))
auc, oof_lgb3, its3 = cv_lgb(BASE_P, 'X3_base', X3); results['LightGBM (v3 features)'] = auc; print('LightGBM v3 features: %.4f  (best rounds %d)' % (auc, its3))

In [ ]:
# --- XGBoost ---
XGB_P = dict(objective='binary:logistic', eval_metric='auc', tree_method='hist', eta=0.04, max_depth=3, min_child_weight=20, subsample=0.8,
             colsample_bytree=0.3, reg_lambda=30, reg_alpha=1, max_cat_to_onehot=8, max_bin=128, nthread=-1)
oof_xgb = np.full(len(y), np.nan)
for (A, B), (a, b) in zip(fold_data(X2, 'X2_base', TE_BASE), folds):
    dA, dB = xgb.DMatrix(A, label=y[a], enable_categorical=True), xgb.DMatrix(B, label=y[b], enable_categorical=True)
    m = xgb.train(XGB_P, dA, CAP, evals=[(dB, 'val')], early_stopping_rounds=100, verbose_eval=False)
    oof_xgb[b] = m.predict(dB, iteration_range=(0, m.best_iteration + 1))
results['XGBoost'] = cv_auc(oof_xgb); print('XGBoost: %.4f' % results['XGBoost'])

In [ ]:
# --- CatBoost (native categorical handling, raw identifiers) ---
Xc = X2.copy(); cat_c = [c for c in Xc if str(Xc[c].dtype) == 'category']
for c in cat_c:
    Xc[c] = Xc[c].astype(object).where(Xc[c].notna(), 'missing').astype(str)
Xc_tr = Xc.loc[tr.encounter_id]
CB_P = dict(iterations=CAP, learning_rate=0.06, depth=5, l2_leaf_reg=10, rsm=0.3, border_count=64, eval_metric='AUC', od_type='Iter', od_wait=100,
            random_seed=0, thread_count=-1, verbose=0, cat_features=cat_c)
oof_cb = np.full(len(y), np.nan)
for a, b in folds:
    m = CatBoostClassifier(**CB_P).fit(Pool(Xc_tr.iloc[a], y[a], cat_features=cat_c), eval_set=Pool(Xc_tr.iloc[b], y[b], cat_features=cat_c))
    oof_cb[b] = m.predict_proba(Xc_tr.iloc[b])[:, 1]
results['CatBoost'] = cv_auc(oof_cb); print('CatBoost: %.4f' % results['CatBoost'])
del Xc, Xc_tr

In [ ]:
# --- models that need imputing / scaling: fitted on the training part of each fold only ---
Xl = X2.drop(columns=['attending_id']); cat_l = [c for c in Xl if str(Xl[c].dtype) == 'category']; num_l = [c for c in Xl if c not in cat_l]
Xl[cat_l] = Xl[cat_l].astype(object); Xl_tr = Xl.loc[tr.encounter_id]


def lin_pre(scaler='standard'):
    sc = QuantileTransformer(n_quantiles=200, output_distribution='normal') if scaler == 'quantile' else StandardScaler()
    return ColumnTransformer([
        ('n', Pipeline([('i', SimpleImputer(strategy='median', add_indicator=True)), ('s', sc)]), num_l),
        ('c', Pipeline([('i', SimpleImputer(strategy='constant', fill_value='missing')),
                        ('o', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=300, sparse_output=False))]), cat_l)])


oof_lr, oof_nb, oof_dt = (np.full(len(y), np.nan) for _ in range(3))
for a, b in folds:
    p = lin_pre('standard').fit(Xl_tr.iloc[a]); A, B = p.transform(Xl_tr.iloc[a]), p.transform(Xl_tr.iloc[b])
    oof_lr[b] = LogisticRegression(C=0.02, max_iter=500).fit(A, y[a]).predict_proba(B)[:, 1]
    oof_dt[b] = DecisionTreeClassifier(max_depth=5, min_samples_leaf=200, random_state=0).fit(A, y[a]).predict_proba(B)[:, 1]
    p = lin_pre('quantile').fit(Xl_tr.iloc[a]); A, B = p.transform(Xl_tr.iloc[a]), p.transform(Xl_tr.iloc[b])
    sel = SelectKBest(f_classif, k=80).fit(A, y[a])
    oof_nb[b] = GaussianNB(var_smoothing=1e-6).fit(sel.transform(A), y[a]).predict_proba(sel.transform(B))[:, 1]
results['Logistic regression'] = cv_auc(oof_lr); results['Naive Bayes (80 best columns)'] = cv_auc(oof_nb); results['Decision tree'] = cv_auc(oof_dt)
print({k: round(v, 4) for k, v in results.items()})

In [ ]:
tab = pd.Series(results).sort_values(ascending=False).round(4).rename('CV ROC-AUC').to_frame()
tab

Other families (random forest, ExtraTrees, HistGradientBoosting, MLP, KNN, LDA, linear SVM) were run with the scripts in the repository; all scored between 0.696 and 0.749, below the boosters. Section 5 tunes the strongest model.

## 5. Tune LightGBM regularisation and target encoding

The only change that raised CV on this data was making LightGBM simpler (15 leaves 0.7494, 6 leaves 0.7519, 4 leaves 0.7521). Here we try simpler and smoother settings, and more target-encoded columns.

In [ ]:
GRID = {
    'v2 setting (baseline)': {},
    '3 leaves': dict(num_leaves=3),
    '2 leaves, lr 0.05': dict(num_leaves=2, learning_rate=0.05),
    'min leaf 800': dict(min_child_samples=800),
    'L2 = 150': dict(lambda_l2=150),
    'L1 = 5, path smooth 50': dict(lambda_l1=5, path_smooth=50),
    '6 leaves, min leaf 800, L2 100, lr 0.03': dict(num_leaves=6, min_child_samples=800, lambda_l2=100, learning_rate=0.03),
    '10% of columns per tree': dict(feature_fraction=0.1),
    '40% of columns per tree': dict(feature_fraction=0.4),
}
if QUICK:
    GRID = dict(list(GRID.items())[:2])
grid_res = {}
for name, over in GRID.items():
    t_ = time.time(); auc, _, its = cv_lgb(dict(BASE_P, **over), 'X2_base', X2)
    grid_res[name] = (auc, its); print('%-45s CV %.4f  rounds %4d  (%ds)' % (name, auc, its, time.time() - t_), flush=True)
best_name = max(grid_res, key=lambda k: grid_res[k][0]); BEST_P = dict(BASE_P, **GRID[best_name])
print('\nbest setting:', best_name, '->', round(grid_res[best_name][0], 4), '(v2 baseline %.4f)' % grid_res['v2 setting (baseline)'][0])

In [ ]:
# more target-encoded columns with the best setting
auc_ext, _, _ = cv_lgb(BEST_P, 'X2_ext', X2, TE_EXT)
print('target encoding on %d columns: CV %.4f | on 4 columns: CV %.4f' % (len(TE_EXT), auc_ext, grid_res[best_name][0]))
USE_COLS = TE_EXT if auc_ext > grid_res[best_name][0] + 0.0002 else TE_BASE
print('using target encoding on:', USE_COLS)

## 6. Final models and blend

The final LightGBM models use the tuned setting on feature sets v2 and v3. They are blended with the logistic regression by averaging ranks (ROC-AUC only looks at the ranking). Blend weights come from the out-of-fold predictions.

In [ ]:
auc2, oof2, its2 = cv_lgb(BEST_P, 'X2_final', X2, USE_COLS)
auc3, oof3, its3 = cv_lgb(BEST_P, 'X3_final', X3, USE_COLS)
print('LightGBM v2 features %.4f | v3 features %.4f' % (auc2, auc3))
r = lambda v: rankdata(v) / len(v)
ok = ~np.isnan(oof2) & ~np.isnan(oof3) & ~np.isnan(oof_lr)
grid = []
for w1 in np.arange(0, 1.01, 0.1):
    for w2 in np.arange(0, 1.01 - w1, 0.1):
        grid.append((roc_auc_score(y[ok], w1 * r(oof2[ok]) + w2 * r(oof3[ok]) + (1 - w1 - w2) * r(oof_lr[ok])), round(w1, 1), round(w2, 1), round(1 - w1 - w2, 1)))
grid.sort(reverse=True); print('top blends (CV AUC, w LightGBM v2, w LightGBM v3, w logistic regression):'); print(grid[:5])
BLEND_CV, W2, W3, WL = grid[0]
print('\nblend CV %.4f | previous best submission (v6) blend CV 0.7527' % BLEND_CV)

In [ ]:
# ---- fit on all training stays and predict the test set ----
t_ = time.time(); n2, n3 = (150, 150) if QUICK else (int(its2 * 1.1), int(its3 * 1.1))
p2, p3 = np.zeros(len(te)), np.zeros(len(te))
for s in range(SEEDS):
    for Xf, n, store in [(X2, n2, 'p2'), (X3, n3, 'p3')]:
        A, (B,) = add_te(Xf.loc[tr.encounter_id], y, [Xf.loc[te.encounter_id]], USE_COLS, seed=s); A, B = prep(A), prep(B)
        pr = lgb.train(dict(BEST_P, seed=s), lgb.Dataset(A, y), n).predict(B) / SEEDS
        if store == 'p2': p2 += pr
        else: p3 += pr
    print('seed', s, 'done', round(time.time() - t_), 's', flush=True)
pl = lin_pre('standard').fit(Xl_tr)
p_lr = LogisticRegression(C=0.02, max_iter=500).fit(pl.transform(Xl_tr), y).predict_proba(pl.transform(Xl.loc[te.encounter_id]))[:, 1]
blend = W2 * r(p2) + W3 * r(p3) + WL * r(p_lr)
order = rankdata(blend, method='ordinal').astype(int) - 1
final = np.sort(p2)[order]                       # blend ranking on the LightGBM probability scale
sub = pd.DataFrame({'encounter_id': te.encounter_id, 'readmit_30d': final})
ss = pd.read_csv(D + 'sample_submission.csv')
assert list(sub.columns) == list(ss.columns) and set(sub.encounter_id) == set(ss.encounter_id) and sub.readmit_30d.between(0, 1).all()
sub.to_csv(OUT + '/submission.csv', index=False)
print('written', OUT + '/submission.csv', sub.shape, '| mean predicted probability %.4f' % sub.readmit_30d.mean())

In [ ]:
summary = dict(cv_models={k: round(v, 4) for k, v in results.items()}, tuning={k: round(v[0], 4) for k, v in grid_res.items()},
               best_setting=best_name, target_encoding_columns=len(USE_COLS), blend_weights=dict(lgb_v2=W2, lgb_v3=W3, logreg=WL), blend_cv=round(BLEND_CV, 4))
json.dump(summary, open(OUT + '/run_summary.json', 'w'), indent=1); print(json.dumps(summary, indent=1))
print('\nSubmit submission.csv only if the blend CV is clearly above 0.7527 (the CV of the v6 blend that scored 0.74685).')